In [ ]:
%pip install requests

In [ ]:
import pandas as pd
import numpy as np
import warnings
import os
import requests
import zipfile
from io import BytesIO

# SDV Imports
from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.single_table import CTGANSynthesizer
from sdv.evaluation.single_table import evaluate_quality

# Suppress technical warnings for a clean presentation output
warnings.filterwarnings('ignore')

def get_uci_bank_data():
    """
    Downloads and prepares the UCI Bank Marketing dataset automatically.
    This is the "Real World Data" for the presentation.
    """
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank.zip"
    csv_filename = 'data/bank/bank.csv'
    
    if not os.path.exists(csv_filename):
        print("-> Downloading UCI Bank Marketing Dataset...")
        response = requests.get(url)
        with zipfile.ZipFile(BytesIO(response.content)) as z:
            # The 'bank.csv' in the zip uses ';' as a separator
            with z.open('bank.csv') as f:
                df = pd.read_csv(f, sep=';')
                # Save it as a standard comma-separated CSV for future use
                df.to_csv(csv_filename, index=False)
    else:
        # If file exists, check if it's semicolon or comma separated
        # We try to be robust here
        try:
            df = pd.read_csv(csv_filename, sep=None, engine='python')
        except Exception:
            df = pd.read_csv(csv_filename)
    
    # Clean column names (UCI data often has quotes in headers)
    df.columns = [c.strip().replace('"', '').replace("'", "") for c in df.columns]
    
    return df

def run_banking_synthesis_presentation():
    print("==========================================================")
    print("  PRESENTATION: REAL-WORLD BANKING DATA SYNTHESIS (SDV)   ")
    print("==========================================================\n")

    # 1. LOAD REAL DATA
    real_data = get_uci_bank_data()
    
    # Verify columns exist to prevent KeyError
    required_cols = ['age', 'job', 'marital', 'balance', 'housing', 'loan']
    available_cols = [c for c in required_cols if c in real_data.columns]
    
    print(f"[Step 1] Real Data Loaded. Shape: {real_data.shape}")
    if len(available_cols) > 0:
        print("Preview of sensitive banking attributes:")
        print(real_data[available_cols].head(3))
    else:
        print("Warning: Could not find standard columns. Available columns are:")
        print(list(real_data.columns))
        
    print("-" * 60)

    # 2. METADATA DETECTION
    print("[Step 2] Creating Metadata (Statistical DNA)...")
    metadata = Metadata.detect_from_dataframe(
        data=real_data, 
        table_name='bank_marketing_data'
    )
    print("-> Metadata successfully mapped categories and distributions.")
    print("-" * 60)

    # 3. CHOOSE & TRAIN THE SYNTHESIZER
    print("[Step 3] Training the Synthesizer (Fitting to Real Data)...")
    #synthesizer = GaussianCopulaSynthesizer(metadata)
    synthesizer = CTGANSynthesizer(metadata)
    synthesizer.fit(real_data)
    print("-> Training Complete. Patterns learned.")
    print("-" * 60)

    # 4. GENERATE SYNTHETIC DATA
    print("[Step 4] Generating 1,000 Synthetic Records...")
    synthetic_data = synthesizer.sample(num_rows=1000)
    
    # Use available columns for preview
    if len(available_cols) > 0:
        print("Synthetic Data Results (Zero Real Customers):")
        print(synthetic_data[available_cols].head(3))
    
    print("-" * 60)

    # 5. QUALITY EVALUATION
    print("[Step 5] Evaluating Statistical Fidelity...")
    quality_report = evaluate_quality(
        real_data,
        synthetic_data,
        metadata
    )
    
    score = quality_report.get_score()
    print(f"\nOVERALL QUALITY SCORE: {score * 100:.2f}%")
    
    # Logic check for the audience
    if 'balance' in real_data.columns:
        print("\nPresentation Logic Check:")
        real_avg_bal = real_data['balance'].mean()
        synth_avg_bal = synthetic_data['balance'].mean()
        print(f"-> Real Avg Balance: ${real_avg_bal:.2f}")
        print(f"-> Synth Avg Balance: ${synth_avg_bal:.2f}")
    
    # Save the output
    synthetic_data.to_csv('synthetic_bank_output.csv', index=False)
    print("\n[SUCCESS] Exported 'synthetic_bank_output.csv' for the demo.")

if __name__ == "__main__":
    run_banking_synthesis_presentation()

  PRESENTATION: REAL-WORLD BANKING DATA SYNTHESIS (SDV)   

[Step 1] Real Data Loaded. Shape: (4521, 17)
Preview of sensitive banking attributes:
   age         job  marital  balance housing loan
0   30  unemployed  married     1787      no   no
1   33    services  married     4789     yes  yes
2   35  management   single     1350     yes   no
------------------------------------------------------------
[Step 2] Creating Metadata (Statistical DNA)...
-> Metadata successfully mapped categories and distributions.
------------------------------------------------------------
[Step 3] Training the Synthesizer (Fitting to Real Data)...
-> Training Complete. Patterns learned.
------------------------------------------------------------
[Step 4] Generating 1,000 Synthetic Records...
Synthetic Data Results (Zero Real Customers):
   age         job  marital  balance housing loan
0   30     student  married     1768     yes   no
1   41  technician  married     2792     yes   no
2   27  management